In [1]:
%load_ext autoreload
%autoreload 2
import sys

sys.path.insert(0, '..')  # nbs/notebook_cache.py
from notebook_cache import NotebookCache

cache = NotebookCache.here()

# Verne, small, step 0: the whole model, dumb, on T1 and T1-wide

What each piece is and how shapes flow through it: `00_explained.ipynb`.

Verne V0.5 (`../verne-v0.5`): a video transformer looks at the cameras **once**, a narrow action transformer reads its keys and values at every layer and denoises an action chunk.

The code lives in `dream_robot/policies/verne/`; this notebook only runs it and looks. Every box below is the dumbest version; later notebooks open one box each.

```-
top frame ─► 16px patches ─► VIDEO transformer (4 layers, 256 wide) ── k,v per layer ─┐
joints ────► one more token ─┘   runs once per chunk                                   │
noise [16 x 8] ─► ACTION transformer (4 layers, 128 wide) ◄────────────────────────────┘
                  10 flow steps ─► 16 joint actions
```

Every score: training seeds `TRAIN_SEEDS` (0 first, then 1, 2) × 50 rollout seeds (1000-1049), at n_exec 8 and 16. Same data as dreamzero's WAMz (10 labelled T1 demos): WAMz got 18 + 16 = 34/40, one model on seeds 1000-1019.

### step 0.1 — the data


In [2]:
import numpy as np
from frozendict import frozendict

from dream_robot.policies.verne.data import load_table, load_top, to_gpu

NAME = 'pick_place_cube_n100'

In [3]:
@cache
def table():
    state, action, episode = load_table(NAME)  # [N, 8], [N, 8], [N]
    return frozendict(state=state, action=action, episode=episode)

table: cached, 1 call(s) stored


In [4]:
@cache
def images():
    return load_top(NAME)  # [N, 3, 128, 128] uint8, a few minutes the first time

images: cached, 1 call(s) stored


In [5]:
data = table()
val = np.random.default_rng(0).permutation(100)[:10]  # as dreamzero / uwm2
rest = np.setdiff1d(np.arange(100), val)
split = frozendict(val=val, labelled=np.random.default_rng(1).choice(rest, 10, replace=False))
demos = to_gpu(images(), data['state'], data['action'], data['episode'], split['labelled'])
print({k: len(v) for k, v in split.items()}, tuple(demos.top.shape))

{'val': 10, 'labelled': 10} (21295, 3, 128, 128)


### step 0.2 — the model


In [6]:
import torch

from dream_robot.policies.verne.model import Verne

net = Verne()
print(f'{sum(p.numel() for p in net.parameters()) / 1e6:.2f}M params')

3.66M params


### step 0.3 — train (flow matching on the 16-action chunk, 20k steps, ~3 min each), each training seed


In [7]:
import dream_robot.policies.verne as verne


@cache
def train_run(steps: int, seed: int):
    return verne.train.train(demos, split['labelled'], split['val'], steps, seed=seed)

train_run: cached, 3 call(s) stored


In [8]:
TRAIN_SEEDS = (0, 1, 2)  # results are cached per seed, so added seeds just join

In [9]:
runs = {seed: train_run(20000, seed) for seed in TRAIN_SEEDS}  # seed -> {'weights', 'log'}

step      0   train 2.026   val 3.142
step   1000   train 0.158   val 0.207
step   2000   train 0.073   val 0.179
step   3000   train 0.087   val 0.269
step   4000   train 0.040   val 0.177
step   5000   train 0.054   val 0.168
step   6000   train 0.057   val 0.244
step   7000   train 0.029   val 0.197
step   8000   train 0.037   val 0.189
step   9000   train 0.033   val 0.216
step  10000   train 0.038   val 0.249
step  11000   train 0.036   val 0.228
step  12000   train 0.037   val 0.183
step  13000   train 0.026   val 0.219
step  14000   train 0.045   val 0.193
step  15000   train 0.036   val 0.206
step  16000   train 0.027   val 0.201
step  17000   train 0.019   val 0.184
step  18000   train 0.020   val 0.207
step  19000   train 0.036   val 0.241
step  20000   train 0.010   val 0.246
step      0   train 2.034   val 2.808
step   1000   train 0.096   val 0.183
step   2000   train 0.045   val 0.164
step   3000   train 0.077   val 0.179
step   4000   train 0.049   val 0.171
step   5000 

### step 0.4 — rollouts: T1 seeds 1000-1049, replan after 8 or all 16 actions, each training seed


In [10]:
from dream_robot.policies.verne import rollouts

env = rollouts.make_env(wide=False)


def total(scores: dict, name: str) -> None:
    # scores: (train seed, n_exec) -> rollout results; prints one line per n_exec
    for n_exec in (8, 16):
        oks = [sum(ok for ok, _, _ in scores[seed, n_exec].values()) for seed in TRAIN_SEEDS]
        n = len(rollouts.SEEDS) * len(TRAIN_SEEDS)
        print(f'{name} n_exec {n_exec:2d}: {" + ".join(map(str, oks))} = {sum(oks)}/{n}')

In [11]:
@cache
def sweep(seed: int, n_exec: int) -> dict[int, tuple[bool, str, int]]:
    return rollouts.sweep(rollouts.load(runs[seed]['weights']), demos, env, n_exec)

sweep: cached, 6 call(s) stored


In [12]:
scores = {(seed, n): sweep(seed, n) for seed in TRAIN_SEEDS for n in (8, 16)}
total(scores, 'T1')

n_exec  8: 24/50   failures {'no_grasp': 23, 'wrong_target': 2, 'dropped': 1}
n_exec 16: 23/50   failures {'no_grasp': 23, 'dropped': 3, 'wrong_target': 1}
n_exec  8: 17/50   failures {'no_grasp': 32, 'dropped': 1}
n_exec 16: 18/50   failures {'no_grasp': 29, 'dropped': 2, 'wrong_target': 1}
n_exec  8: 13/50   failures {'no_grasp': 36, 'wrong_target': 1}
n_exec 16: 18/50   failures {'no_grasp': 31, 'dropped': 1}
T1 n_exec  8: 24 + 17 + 13 = 54/150
T1 n_exec 16: 23 + 18 + 18 = 59/150


**T1, train seeds 0 + 1 + 2: 24 + 17 + 13 = 54/150 at n_exec 8, 23 + 18 + 18 = 59/150 at 16** (36%, 39%; WAMz 18 + 16 = 34/40, one model, seeds 1000-1019 only). Failures mostly no_grasp.

- Seeds spread 13-24 of 50 at n_exec 8: wider than n_exec 8 vs 16. Held-out loss at the end 0.246, 0.196, 0.201 doesn't predict the order (seed 0 has the worst loss, the best score).
- On seeds 1000-1019 alone, seed 0 gives 8 and 9, the scores this notebook showed before `rollouts.py`: the module reproduces them.


### step 0.5 — T1-wide: a cube that moves (x ±10 cm, y −10..+6 cm)

On T1 the cube spawns within ±3 cm, so a policy can grasp the average spot without looking. T1-wide (`data/robosuite/pick_place_cube_wide_n100`, expert 100/100) makes it find the cube. 90 train demos, 10 held out (the same episode ids as T1's val), 20k steps, each training seed, rollout seeds 1000-1049.


In [13]:
WIDE = 'pick_place_cube_wide_n100'


@cache
def table_wide():
    state, action, episode = load_table(WIDE)  # [N, 8], [N, 8], [N]
    return frozendict(state=state, action=action, episode=episode)

table_wide: cached, 1 call(s) stored


In [14]:
@cache
def images_wide():
    return load_top(WIDE)  # [N, 3, 128, 128] uint8, a few minutes the first time

images_wide: cached, 1 call(s) stored


In [15]:
w_data = table_wide()
w_split = frozendict(val=val, train=np.setdiff1d(np.arange(100), val))
w_demos = to_gpu(
    images_wide(), w_data['state'], w_data['action'], w_data['episode'], w_split['train']
)
print({k: len(v) for k, v in w_split.items()}, tuple(w_demos.top.shape))

{'val': 10, 'train': 90} (22409, 3, 128, 128)


In [16]:
@cache
def train_wide(steps: int, seed: int):
    return verne.train.train(w_demos, w_split['train'], w_split['val'], steps, seed=seed)

train_wide: cached, 3 call(s) stored


In [17]:
w_runs = {seed: train_wide(20000, seed) for seed in TRAIN_SEEDS}

step      0   train 2.087   val 2.870
step   1000   train 0.194   val 0.134
step   2000   train 0.073   val 0.090
step   3000   train 0.103   val 0.095
step   4000   train 0.045   val 0.081
step   5000   train 0.060   val 0.072
step   6000   train 0.080   val 0.079
step   7000   train 0.038   val 0.063
step   8000   train 0.049   val 0.070
step   9000   train 0.023   val 0.059
step  10000   train 0.047   val 0.072
step  11000   train 0.042   val 0.061
step  12000   train 0.044   val 0.054
step  13000   train 0.044   val 0.069
step  14000   train 0.054   val 0.052
step  15000   train 0.039   val 0.053
step  16000   train 0.039   val 0.050
step  17000   train 0.027   val 0.058
step  18000   train 0.027   val 0.049
step  19000   train 0.047   val 0.053
step  20000   train 0.024   val 0.055
step      0   train 2.036   val 3.129
step   1000   train 0.104   val 0.140
step   2000   train 0.086   val 0.098
step   3000   train 0.099   val 0.095
step   4000   train 0.067   val 0.089
step   5000 

In [18]:
w_env = rollouts.make_env(wide=True)

In [19]:
@cache
def sweep_wide(seed: int, n_exec: int) -> dict[int, tuple[bool, str, int]]:
    return rollouts.sweep(rollouts.load(w_runs[seed]['weights']), w_demos, w_env, n_exec)

sweep_wide: cached, 6 call(s) stored


In [20]:
w_scores = {(seed, n): sweep_wide(seed, n) for seed in TRAIN_SEEDS for n in (8, 16)}
total(w_scores, 'T1-wide')

n_exec  8:  3/50   failures {'no_grasp': 47}
n_exec 16:  3/50   failures {'no_grasp': 47}
n_exec  8:  1/50   failures {'no_grasp': 49}
n_exec 16:  1/50   failures {'no_grasp': 49}
n_exec  8:  1/50   failures {'no_grasp': 49}
n_exec 16:  1/50   failures {'no_grasp': 48, 'wrong_target': 1}
T1-wide n_exec  8: 3 + 1 + 1 = 5/150
T1-wide n_exec 16: 3 + 1 + 1 = 5/150


**T1-wide, train seeds 0 + 1 + 2: 3 + 1 + 1 = 5/150 at both n_exec**, nearly all no_grasp. Held-out loss 0.055, 0.050, 0.047 (90 demos, no overfit), so the low score isn't overfitting.


### step 0.6 — the close-point probe: does the gripper close where the cube is?

For each T1-wide rollout (n_exec 8): the gripper's xy when it first commands close, against the cube's spawn xy. Slope ≈ 1 and a small miss: it goes to the cube. Slope ≈ 0: it goes to a remembered spot whatever the cube does. One line per training seed.


In [21]:
@cache
def close_points_wide(seed: int, n_exec: int) -> dict:
    return rollouts.close_points(rollouts.load(w_runs[seed]['weights']), w_demos, w_env, n_exec)

close_points_wide: cached, 3 call(s) stored


In [22]:
fits = {}
for seed in TRAIN_SEEDS:
    print(f'train seed {seed}: ', end='')
    fits[seed] = rollouts.close_fit(close_points_wide(seed, 8))

train seed 0: closes 50/50, miss 10.9 cm, x slope -0.31 (corr -0.33, intercept +3.7 cm), y slope -0.09 (corr -0.09, intercept -2.4 cm)
train seed 1: closes 50/50, miss 10.3 cm, x slope -0.13 (corr -0.12, intercept +3.2 cm), y slope -0.20 (corr -0.28, intercept -1.9 cm)
train seed 2: closes 50/50, miss 10.0 cm, x slope -0.17 (corr -0.16, intercept +3.1 cm), y slope +0.05 (corr +0.06, intercept -0.7 cm)


**Probe: blind, all three seeds.** Each closes in 50/50 runs, but where it closes doesn't follow the cube:

- seed 0: x slope −0.31, y −0.09; intercepts x +3.7, y −2.4 cm; mean miss 10.9 cm
- seed 1: x slope −0.13, y −0.20; intercepts x +3.2, y −1.9 cm; mean miss 10.3 cm
- seed 2: x slope −0.17, y +0.05; intercepts x +3.1, y −0.7 cm; mean miss 10.0 cm

It goes to remembered spots, whatever the cube does: with slope ≈ 0 the intercept is that spot, about x +3 cm, y −1 to −2 cm.
